In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report)


In [2]:
df = pd.read_csv("../data/heart_failure_clinical_records_dataset.csv")

In [3]:
print("Shape:", df.shape)

Shape: (299, 13)


In [4]:
print("Missing values:", df.isna().sum().sum())

Missing values: 0


In [5]:
print(df["DEATH_EVENT"].value_counts())

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


In [7]:
X = df.drop(columns="DEATH_EVENT")
y = df["DEATH_EVENT"]
features = list(X.columns)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


In [9]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

,steps,"[('standardscaler', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [10]:
y_pred = model.predict(X_test)


In [11]:
y_pred = model.predict(X_test)

In [12]:
acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", round(acc, 4))
print(classification_report(y_test, y_pred))

Test accuracy: 0.8167
              precision    recall  f1-score   support

           0       0.83      0.93      0.87        41
           1       0.79      0.58      0.67        19

    accuracy                           0.82        60
   macro avg       0.81      0.75      0.77        60
weighted avg       0.81      0.82      0.81        60



In [13]:
confusion_matrix(y_test, y_pred)

array([[38,  3],
       [ 8, 11]])

In [15]:
import gradio as gr

DEFAULTS = {'age':60,'anaemia':0,'creatinine_phosphokinase':250,'diabetes':0,
            'ejection_fraction':38,'high_blood_pressure':0,'platelets':263000,
            'serum_creatinine':1.1,'serum_sodium':137,'sex':1,'smoking':0,'time':100}
POS, NEG = "Death event likely", "Survival likely"

def predict_gradio(*values):
    row = pd.DataFrame([list(values)], columns=features)
    p = model.predict_proba(row)[0][1]
    return {POS: float(p), NEG: float(1 - p)}

inputs = [gr.Number(label=f, value=DEFAULTS.get(f, 0)) for f in features]

demo = gr.Interface(
    fn=predict_gradio, inputs=inputs, outputs=gr.Label(num_top_classes=2),
    title="Heart Failure Survival Predictor",
    description=f"Logistic Regression | test accuracy: {acc*100:.1f}%. "
                "Binary columns (anaemia, diabetes, high_blood_pressure, sex, smoking) take 0 or 1.")
demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
from flask import Flask, request, render_template_string
from werkzeug.serving import make_server
import threading

try:
    flask_server.shutdown()      # stops the old server if you re-run this cell
except NameError:
    pass

app = Flask(__name__)

PAGE = """
<!doctype html>
<title>Heart Failure Predictor</title>
<style>
  body{font-family:Arial;max-width:520px;margin:30px auto;padding:0 12px}
  label{display:block;margin-top:10px;font-weight:bold}
  input{width:100%;padding:7px;box-sizing:border-box}
  button{margin-top:16px;padding:10px 18px;cursor:pointer}
  .res{margin-top:20px;padding:14px;background:#eef;border-radius:6px}
</style>
<h2>Heart Failure Survival Predictor</h2>
<p>Logistic Regression &middot; test accuracy {{ acc }}%</p>
<form method="post">
  {% for f in features %}
    <label>{{ f }}</label>
    <input type="number" step="any" name="{{ f }}" value="{{ vals[f] }}" required>
  {% endfor %}
  <button type="submit">Predict</button>
</form>
{% if result %}
  <div class="res"><b>{{ result }}</b><br>Probability: {{ prob }}%</div>
{% endif %}
"""

@app.route("/", methods=["GET", "POST"])
def index():
    vals = {f: DEFAULTS.get(f, 0) for f in features}
    result = prob = None
    if request.method == "POST":
        vals = {f: float(request.form[f]) for f in features}
        p = model.predict_proba(pd.DataFrame([[vals[f] for f in features]], columns=features))[0][1]
        result = POS if p >= 0.5 else NEG
        prob = round(max(p, 1 - p) * 100, 2)
    return render_template_string(PAGE, features=features, vals=vals,
                                  result=result, prob=prob, acc=round(acc * 100, 1))

flask_server = make_server("127.0.0.1", 5000, app)
threading.Thread(target=flask_server.serve_forever, daemon=True).start()
print("Open http://127.0.0.1:5000 in your browser")
# flask_server.shutdown()   # run this to stop it

Open http://127.0.0.1:5000 in your browser
